In [1]:
import os
import pyspark

print("Répertoire :", os.getcwd())
print("Version PySpark :", pyspark.__version__)

Répertoire : /home/jovyan/work
Version PySpark : 3.5.3


# TP : Mise en oeuvre d’un pipeline ETL et d’un Data Lakehouse (Bronze/Silver/Gold)

## Preparation de l'envirronnement

### 1.1 Initialisation de la session Spark

In [2]:

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Eshop-ETL-Lakehouse")
    .master("local[*]")
    .getOrCreate()
)

print("Spark version :", spark.version)
print("Spark master :", spark.sparkContext.master)
print("Application ID :", spark.sparkContext.applicationId)


Spark version : 3.5.3
Spark master : local[*]
Application ID : local-1794148640001


### 1.2 Ingestion des données sources

In [3]:
customers_raw =(
    spark.read.format("csv")
    .option("header", "true")
    .option("delimiter", ";")
    .option("inferSchema", "false")
    .load("data/source/customers.csv")
)

customers_raw.printSchema()
customers_raw.show(10, truncate=False)

print("Nombre de clients :", customers_raw.count())

root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- region: string (nullable = true)
 |-- registration_date: string (nullable = true)

+-----------+-------------+------+-----------------+
|customer_id|customer_name|region|registration_date|
+-----------+-------------+------+-----------------+
|C001       |Alice 01     |Nord  |2024-01-08       |
|C002       |Youssef 02   |Sud   |2024-01-15       |
|C003       |Sara 03      |Centre|2024-01-22       |
|C004       |Omar 04      |Est   |2024-01-29       |
|C005       |Nadia 05     |Ouest |2024-02-05       |
|C006       |Amine 06     |Nord  |2024-02-12       |
|C007       |Alice 07     |Sud   |2024-02-19       |
|C008       |Youssef 08   |Centre|2024-02-26       |
|C009       |Sara 09      |Est   |2024-03-04       |
|C010       |Omar 10      |Ouest |2024-03-11       |
+-----------+-------------+------+-----------------+
only showing top 10 rows

Nombre de clients : 32


In [4]:
# Let's continue the ingestion process by reading the other data sources
def read_csv_data(filename):
    return (
        spark.read.format("csv")
        .option("header", "true")
        .option("delimiter", ";")
        .option("inferSchema", "false")
        .csv(f"data/source/{filename}")
    )

products_raw = read_csv_data("products.csv")
orders_raw = read_csv_data("orders.csv")
order_items_raw = read_csv_data("order_items.csv")


In [5]:
#promotions file
promotions_raw = spark.read.text("data/source/promotions.jsonl")
promotions_raw.printSchema()
promotions_raw.show(truncate=False)

root
 |-- value: string (nullable = true)

+----------------------------------------------------------------------------------------------------+
|value                                                                                               |
+----------------------------------------------------------------------------------------------------+
|{"promotion_code":"PROMO5","discount_rate":0.05,"start_date":"2025-01-01","end_date":"2025-12-31"}  |
|{"promotion_code":"PROMO10","discount_rate":0.1,"start_date":"2025-05-01","end_date":"2025-05-31"}  |
|{"promotion_code":"SUMMER15","discount_rate":0.15,"start_date":"2025-06-01","end_date":"2025-08-31"}|
|{"promotion_code":"VIP20","discount_rate":0.2,"start_date":"2025-01-01","end_date":"2025-12-31"}    |
|{"promotion_code":"OLD30","discount_rate":0.3,"start_date":"2024-01-01","end_date":"2024-12-31"}    |
|{"promotion_code":"BAD_JSON","discount_rate":0.10,"start_date":"2025-06-01","end_date":             |
+-----------------------------

In [6]:
# number of lines in each source
sources = {
    "customers": customers_raw,
    "products": products_raw,
    "orders": orders_raw,
    "order_items": order_items_raw,
    "promotions": promotions_raw
}

for name, df in sources.items():
    print(f"{name}: {df.count()} lignes")


customers: 32 lignes
products: 14 lignes
orders: 44 lignes
order_items: 86 lignes
promotions: 6 lignes


### Data Exploration

In [7]:
for name, df in sources.items():
    print(f"\n===== {name.upper()} =====")

    df.printSchema()
    df.show(5, truncate=False)


===== CUSTOMERS =====
root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- region: string (nullable = true)
 |-- registration_date: string (nullable = true)

+-----------+-------------+------+-----------------+
|customer_id|customer_name|region|registration_date|
+-----------+-------------+------+-----------------+
|C001       |Alice 01     |Nord  |2024-01-08       |
|C002       |Youssef 02   |Sud   |2024-01-15       |
|C003       |Sara 03      |Centre|2024-01-22       |
|C004       |Omar 04      |Est   |2024-01-29       |
|C005       |Nadia 05     |Ouest |2024-02-05       |
+-----------+-------------+------+-----------------+
only showing top 5 rows


===== PRODUCTS =====
root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- unit_price: string (nullable = true)
 |-- unit_cost: string (nullable = true)

+----------+------------+------------+----------+--------

In [8]:
# Volumes verification

expected_counts = {
    "customers": 32,
    "products": 14,
    "orders": 44,
    "order_items": 86,
    "promotions": 6
}

for name, df in sources.items():
    actual = df.count()
    expected = expected_counts[name]

    print(
        f"{name}: attendu={expected}, "
        f"obtenu={actual}"
    )

    assert actual == expected, (
        f"Volume incorrect pour {name}"
    )

customers: attendu=32, obtenu=32
products: attendu=14, obtenu=14
orders: attendu=44, obtenu=44
order_items: attendu=86, obtenu=86
promotions: attendu=6, obtenu=6



### Conclusion de l'ingestion

Les cinq sources ont été chargées dans des DataFrames Spark.

Les champs CSV ont été conservés sous forme de chaînes
de caractères afin de préserver leurs valeurs originales.

Le fichier JSONL a été lu en mode texte pour conserver
également les lignes mal formées.

Les volumes ont été comparés aux valeurs de référence.

Les contrôles de qualité, les conversions de types et la
gestion des anomalies seront réalisés dans la couche Silver.


# Bronze Zone

In [9]:
# Original files consservation 

from pathlib import Path
import shutil

project_dir = Path("/home/jovyan/work")

source_dir = project_dir / "data" / "source"
bronze_dir = project_dir / "data_lakehouse" / "bronze" / "original"

bronze_dir.mkdir(parents=True, exist_ok=True)

filenames = [
    "customers.csv",
    "products.csv",
    "orders.csv",
    "order_items.csv",
    "promotions.jsonl"
]

for filename in filenames:
    source_file = source_dir / filename
    destination_file = bronze_dir / filename

    shutil.copy2(source_file, destination_file)

    print(f"Copié : {filename}")


Copié : customers.csv
Copié : products.csv
Copié : orders.csv
Copié : order_items.csv
Copié : promotions.jsonl


In [10]:
# Copy intgrity verification

import hashlib

def file_hash(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

for filename in filenames:
    original = source_dir / filename
    bronze = bronze_dir / filename

    assert file_hash(original) == file_hash(bronze)

    print(f"{filename}: copie identique")

customers.csv: copie identique
products.csv: copie identique
orders.csv: copie identique
order_items.csv: copie identique
promotions.jsonl: copie identique


#### Json manifest

In [11]:
from uuid import uuid4
from datetime import datetime, timezone

run_id = str(uuid4())

print("Run ID :", run_id)
print("Date :", datetime.now(timezone.utc).isoformat())

Run ID : 9cf72673-2701-4624-af3b-b5a0b8948fdb
Date : 2026-11-08T14:37:31.075554+00:00


In [12]:
import json
import shutil

manifest = []

for filename in filenames:
    source_path = source_dir / filename
    bronze_path = bronze_dir / filename

    record = {
        "run_id": run_id,
        "source": filename,
        "ingestion_time": datetime.now(timezone.utc).isoformat(),
        "bronze_path": str(bronze_path),
        "row_count": None,
        "status": "failed"
    }

    try:
        shutil.copy2(source_path, bronze_path)

        assert file_hash(source_path) == file_hash(bronze_path)

        if filename.endswith(".jsonl"):
            df = spark.read.text(str(bronze_path))
        else:
            df = (
                spark.read
                .option("header", True)
                .option("sep", ";")
                .option("inferSchema", False)
                .csv(str(bronze_path))
            )

        record["row_count"] = df.count()
        record["sha256"] = file_hash(bronze_path)
        record["status"] = "success"

    except Exception as e:
        record["error"] = str(e)

    manifest.append(record)

    print(filename, record["status"], record["row_count"])


customers.csv success 32
products.csv success 14
orders.csv success 44
order_items.csv success 86
promotions.jsonl success 6


In [13]:
metadata_dir = project_dir / "data_lakehouse" / "metadata" / "ingestion"

metadata_dir.mkdir(parents=True, exist_ok=True)

manifest_path = metadata_dir / f"manifest_{run_id}.json"

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

print("Manifeste enregistré :", manifest_path)

assert all(record["status"] == "success" for record in manifest)

Manifeste enregistré : /home/jovyan/work/data_lakehouse/metadata/ingestion/manifest_9cf72673-2701-4624-af3b-b5a0b8948fdb.json


##### Manifest verification

In [14]:
from pathlib import Path
import json

metadata_dir = Path(
    "/home/jovyan/work/data_lakehouse/metadata/ingestion"
)

manifests = sorted(
    metadata_dir.glob("manifest_*.json"),
    key=lambda p: p.stat().st_mtime
)

if manifests:
    latest_manifest = manifests[-1]

    with open(latest_manifest, "r", encoding="utf-8") as f:
        records = json.load(f)

    for record in records:
        print(
            record["source"],
            record["row_count"],
            record["status"],
            record["run_id"]
        )
else:
    print("Aucun manifeste trouvé.")

customers.csv 32 success 9cf72673-2701-4624-af3b-b5a0b8948fdb
products.csv 14 success 9cf72673-2701-4624-af3b-b5a0b8948fdb
orders.csv 44 success 9cf72673-2701-4624-af3b-b5a0b8948fdb
order_items.csv 86 success 9cf72673-2701-4624-af3b-b5a0b8948fdb
promotions.jsonl 6 success 9cf72673-2701-4624-af3b-b5a0b8948fdb


###  we can read with Spark verification

In [15]:
# test in customers file
customers_bronze = (
    spark.read
    .option("header", True)
    .option("sep", ";")
    .option("inferSchema", False)
    .csv(str(bronze_dir / "customers.csv"))
)

customers_count = customers_bronze.count()

print("Nombre de lignes Bronze :", customers_count)

assert customers_count == 32

Nombre de lignes Bronze : 32


In [16]:
# other files 
for record in manifest:

    filename = record["source"]
    path = record["bronze_path"]
    expected_count = record["row_count"]

    if filename.endswith(".jsonl"):
        df = spark.read.text(path)
    else:
        df = (
            spark.read
            .option("header", True)
            .option("sep", ";")
            .option("inferSchema", False)
            .csv(path)
        )

    actual_count = df.count()

    assert actual_count == expected_count, (
        f"Volume incorrect pour {filename}"
    )

    print(f"{filename} : {actual_count} lignes  OK")

customers.csv : 32 lignes  OK
products.csv : 14 lignes  OK
orders.csv : 44 lignes  OK
order_items.csv : 86 lignes  OK
promotions.jsonl : 6 lignes  OK


## Silver

In [18]:
# Idemtiffy required columns for cusomers
from pyspark.sql import functions as F

required_columns = [
    "customer_id",
    "customer_name",
    "region"
]

for column in required_columns:
    missing_count = customers_bronze.filter(
        F.col(column).isNull()
        | (F.trim(F.col(column)) == "")
    ).count()

    print(f"{column}: {missing_count} valeurs manquantes")

customer_id: 0 valeurs manquantes
customer_name: 1 valeurs manquantes
region: 0 valeurs manquantes


In [19]:
# Invalid dates

customers_typed = customers_bronze.withColumn(
    "registration_date_parsed",
    F.expr("try_cast(registration_date as date)")
)

customers_typed.select(
    "customer_id",
    "registration_date",
    "registration_date_parsed"
).show(truncate=False)

+-----------+-----------------+------------------------+
|customer_id|registration_date|registration_date_parsed|
+-----------+-----------------+------------------------+
|C001       |2024-01-08       |2024-01-08              |
|C002       |2024-01-15       |2024-01-15              |
|C003       |2024-01-22       |2024-01-22              |
|C004       |2024-01-29       |2024-01-29              |
|C005       |2024-02-05       |2024-02-05              |
|C006       |2024-02-12       |2024-02-12              |
|C007       |2024-02-19       |2024-02-19              |
|C008       |2024-02-26       |2024-02-26              |
|C009       |2024-03-04       |2024-03-04              |
|C010       |2024-03-11       |2024-03-11              |
|C011       |2024-03-18       |2024-03-18              |
|C012       |2024-03-25       |2024-03-25              |
|C013       |2024-04-01       |2024-04-01              |
|C014       |2024-04-08       |2024-04-08              |
|C015       |2024-04-15       |

In [20]:
customers_typed.filter(
    F.col("registration_date").isNotNull()
    & F.col("registration_date_parsed").isNull()
).select(
    "customer_id",
    "registration_date",
    "registration_date_parsed"
).show(truncate=False)

+-----------+-----------------+------------------------+
|customer_id|registration_date|registration_date_parsed|
+-----------+-----------------+------------------------+
|C031       |2025-02-30       |NULL                    |
+-----------+-----------------+------------------------+



In [22]:
# Detect duplicates id in customers file 
duplicate_ids = (
    customers_bronze.groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
)
duplicate_ids.show()

+-----------+-----+
|customer_id|count|
+-----------+-----+
|       C005|    2|
+-----------+-----+



In [23]:
# Detect exact duplicates in customers file ( in all columns)
exact_duplicates = (
    customers_bronze
    .groupBy(*customers_bronze.columns)
    .count()
    .filter(F.col("count") > 1)
)

exact_duplicates.show(truncate=False)


+-----------+-------------+------+-----------------+-----+
|customer_id|customer_name|region|registration_date|count|
+-----------+-------------+------+-----------------+-----+
|C005       |Nadia 05     |Ouest |2024-02-05       |2    |
+-----------+-------------+------+-----------------+-----+



In [24]:

# Build quality reasons for customers

customers_checked = customers_typed.withColumn(
    "quality_reason",
    F.concat_ws(
        " | ",
        F.when(
            F.col("customer_id").isNull() |
            (F.trim(F.col("customer_id")) == ""),
            "ID_MANQUANT"
        ),
        F.when(
            F.col("customer_name").isNull() |
            (F.trim(F.col("customer_name")) == ""),
            "NOM_MANQUANT"
        ),
        F.when(
            F.col("region").isNull() |
            (F.trim(F.col("region")) == ""),
            "REGION_MANQUANTE"
        ),
        F.when(
            F.col("registration_date_parsed").isNull(),
            "DATE_INVALIDE"
        )
    )
)

In [25]:
# verification of quality reasons
customers_checked.filter(
    F.col("quality_reason") != ""
).select(
    "customer_id",
    "customer_name",
    "registration_date",
    "quality_reason"
).show(truncate=False)

+-----------+-------------+-----------------+----------------------------+
|customer_id|customer_name|registration_date|quality_reason              |
+-----------+-------------+-----------------+----------------------------+
|C031       |NULL         |2025-02-30       |NOM_MANQUANT | DATE_INVALIDE|
+-----------+-------------+-----------------+----------------------------+



In [26]:
# Mange duplicates in customers 
customers_unique = customers_checked.dropDuplicates(
    customers_bronze.columns
)

print("Avant :", customers_checked.count())
print("Après :", customers_unique.count())

Avant : 32
Après : 31


##### Duplicates audit

In [28]:
customers_duplicates_audit = exact_duplicates.withColumn(
    "extra_copies",
    F.col("count") - 1
)

customers_duplicates_audit.show(truncate=False)

+-----------+-------------+------+-----------------+-----+------------+
|customer_id|customer_name|region|registration_date|count|extra_copies|
+-----------+-------------+------+-----------------+-----+------------+
|C005       |Nadia 05     |Ouest |2024-02-05       |2    |1           |
+-----------+-------------+------+-----------------+-----+------------+



In [29]:
# id conflicts in customers file
customers_conflicts = (
    customers_unique
    .filter(
        F.col("customer_id").isNotNull()
        & (F.trim(F.col("customer_id")) != "")
    )
    .groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
)

customers_conflicts.show(truncate=False)

+-----------+-----+
|customer_id|count|
+-----------+-----+
+-----------+-----+



In [30]:
# Moove invalid customers to quarantine
customers_valid = customers_unique.filter(
    F.col("quality_reason") == ""
)

customers_quarantine = customers_unique.filter(
    F.col("quality_reason") != ""
)

print("Valid customers:", customers_valid.count())
print("Quarantined customers:", customers_quarantine.count())

Valid customers: 30
Quarantined customers: 1


In [31]:
# verification of the quarantine
customers_quarantine.select(
    "customer_id",
    "customer_name",
    "registration_date",
    "quality_reason"
).show(truncate=False)

+-----------+-------------+-----------------+----------------------------+
|customer_id|customer_name|registration_date|quality_reason              |
+-----------+-------------+-----------------+----------------------------+
|C031       |NULL         |2025-02-30       |NOM_MANQUANT | DATE_INVALIDE|
+-----------+-------------+-----------------+----------------------------+



In [33]:
# Data conservation verification
bronze_count = customers_bronze.count()
valid_count = customers_valid.count()
quarantine_count = customers_quarantine.count()

audit_count = (
    customers_duplicates_audit
    .agg(
        F.coalesce(
            F.sum("extra_copies"),
            F.lit(0)
        ).alias("total")
    )
    .first()["total"]
)

total = valid_count + quarantine_count + audit_count

print("Bronze:", bronze_count)
print("Valid:", valid_count)
print("Quarantine:", quarantine_count)
print("Duplicates audit:", audit_count)
print("Total:", total)

assert bronze_count == total

print("Data conservation verified")

Bronze: 32
Valid: 30
Quarantine: 1
Duplicates audit: 1
Total: 32
Data conservation verified


### Save the data 

In [34]:

# Define storage paths

silver_path = project_dir / "data_lakehouse" / "silver" / "customers"

quarantine_path = project_dir / "data_lakehouse" / "quarantine" / "customers"

audit_path = project_dir / "data_lakehouse" / "metadata" / "duplicates" / "customers"

# Save results as Parquet

customers_valid.write.mode("overwrite").parquet(str(silver_path))

customers_quarantine.write.mode("overwrite").parquet(str(quarantine_path))

customers_duplicates_audit.write.mode("overwrite").parquet(str(audit_path))

print("Customers data successfully persisted ")


Customers data successfully persisted 


In [35]:
# Final verification of persisted data
customers_silver_reloaded = spark.read.parquet(
    str(silver_path)
)

customers_quarantine_reloaded = spark.read.parquet(
    str(quarantine_path)
)

customers_audit_reloaded = spark.read.parquet(
    str(audit_path)
)

print("Silver:", customers_silver_reloaded.count())
print("Quarantine:", customers_quarantine_reloaded.count())
print("Audit groups:", customers_audit_reloaded.count())

customers_silver_reloaded.printSchema()

Silver: 30
Quarantine: 1
Audit groups: 1
root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- region: string (nullable = true)
 |-- registration_date: string (nullable = true)
 |-- registration_date_parsed: date (nullable = true)
 |-- quality_reason: string (nullable = true)



#### automate rest of the silver pipeline

In [37]:
from pyspark.sql import functions as F, types as T

def missing(column):
    return F.col(column).isNull() | (F.trim(F.col(column)) == "")

def add_reasons(df, checks):
    return df.withColumn(
        "quality_reason",
        F.concat_ws(" | ", *[
            F.when(condition, F.lit(label))
            for label, condition in checks
        ])
    )

def read_bronze(name):
    return (
        spark.read
        .option("header", True)
        .option("sep", ";")
        .option("inferSchema", False)
        .csv(str(bronze_dir / f"{name}.csv"))
    )

def save_quality(name, df, key, raw_cols):
    # Audit des doublons exacts
    audit = (
        df.groupBy(*raw_cols).count()
        .filter(F.col("count") > 1)
        .withColumn("extra_copies", F.col("count") - 1)
    )

    unique = df.dropDuplicates(raw_cols)

    # Conflits d'identifiants
    conflicts = (
        unique.filter(~missing(key))
        .groupBy(key).count()
        .filter(F.col("count") > 1)
        .select(key)
        .withColumn("is_conflict", F.lit(True))
    )

    checked = (
        unique.join(conflicts, on=key, how="left")
        .withColumn(
            "quality_reason",
            F.concat_ws(
                " | ",
                F.col("quality_reason"),
                F.when(F.col("is_conflict") == True,
                       "CONFLIT_IDENTIFIANT")
            )
        )
        .drop("is_conflict")
    )

    valid = checked.filter(F.col("quality_reason") == "")
    rejected = checked.filter(F.col("quality_reason") != "")

    root = project_dir / "data_lakehouse"

    valid.write.mode("overwrite").parquet(
        str(root / "silver" / name)
    )
    rejected.write.mode("overwrite").parquet(
        str(root / "quarantine" / name)
    )
    audit.write.mode("overwrite").parquet(
        str(root / "metadata" / "duplicates" / name)
    )

    copies = audit.agg(
        F.coalesce(F.sum("extra_copies"), F.lit(0)).alias("n")
    ).first()["n"]

    raw_n = df.count()
    valid_n = valid.count()
    rejected_n = rejected.count()

    assert raw_n == valid_n + rejected_n + copies

    print(
        f"{name}: {raw_n} = {valid_n} Silver + "
        f"{rejected_n} rejets + {copies} copies"
    )

    return valid, rejected, audit

##### products file rules 

In [38]:
p = read_bronze("products")

p = (
    p.withColumn(
        "unit_price_parsed",
        F.expr("try_cast(unit_price as decimal(12,2))")
    )
    .withColumn(
        "unit_cost_parsed",
        F.expr("try_cast(unit_cost as decimal(12,2))")
    )
)

p = add_reasons(p, [
    ("ID_MANQUANT", missing("product_id")),
    ("NOM_MANQUANT", missing("product_name")),
    ("CATEGORIE_MANQUANTE", missing("category")),
    ("PRIX_INVALIDE",
     F.col("unit_price_parsed").isNull() |
     (F.col("unit_price_parsed") < 0)),
    ("COUT_INVALIDE",
     F.col("unit_cost_parsed").isNull() |
     (F.col("unit_cost_parsed") < 0))
])

products_silver, products_quarantine, products_audit = save_quality(
    "products", p, "product_id", read_bronze("products").columns
)

products: 14 = 12 Silver + 1 rejets + 1 copies


In [40]:
# same for promotions 
promo_schema = T.StructType([
    T.StructField("promotion_code", T.StringType()),
    T.StructField("discount_rate", T.StringType()),
    T.StructField("start_date", T.StringType()),
    T.StructField("end_date", T.StringType())
])

pr = spark.read.text(str(bronze_dir / "promotions.jsonl"))

pr = (
    pr.withColumn("parsed", F.from_json("value", promo_schema))
    .withColumn("json_ok", F.col("parsed").isNotNull())
    .select("value", "json_ok", "parsed.*")
    .withColumn(
        "discount_rate_parsed",
        F.expr("try_cast(discount_rate as decimal(6,4))")
    )
    .withColumn(
        "start_date_parsed",
        F.expr("try_cast(start_date as date)")
    )
    .withColumn(
        "end_date_parsed",
        F.expr("try_cast(end_date as date)")
    )
)

pr = add_reasons(pr, [
    ("JSON_MALFORME", ~F.col("json_ok")),
    ("CODE_MANQUANT", missing("promotion_code")),
    ("TAUX_INVALIDE",
     F.col("discount_rate_parsed").isNull() |
     (F.col("discount_rate_parsed") < 0) |
     (F.col("discount_rate_parsed") > 1)),
    ("DATE_DEBUT_INVALIDE", F.col("start_date_parsed").isNull()),
    ("DATE_FIN_INVALIDE", F.col("end_date_parsed").isNull()),
    ("PERIODE_INCOHERENTE",
     F.col("start_date_parsed") > F.col("end_date_parsed"))
])

promotions_silver, promotions_quarantine, promotions_audit = save_quality(
    "promotions", pr, "promotion_code", ["value"]
)

promotions: 6 = 5 Silver + 1 rejets + 0 copies


In [41]:
# orders 

orders = read_bronze("orders")

orders = (
    orders.withColumn(
        "order_date_parsed",
        F.expr("try_cast(order_date as date)")
    )
    .join(
        customers_valid.select("customer_id")
        .withColumn("client_found", F.lit(True)),
        on="customer_id",
        how="left"
    )
)
orders = add_reasons(orders, [
    ("ID_MANQUANT", missing("order_id")),
    ("CLIENT_MANQUANT", missing("customer_id")),
    ("CLIENT_INCONNU",
     ~missing("customer_id") & F.col("client_found").isNull()),
    ("DATE_INVALIDE", F.col("order_date_parsed").isNull()),
    ("STATUT_INVALIDE",
     missing("status") |
     ~F.col("status").isin(
         "PAID", "DELIVERED", "PENDING", "CANCELLED"
     ))
])

ref_promos = promotions_silver.select(
    F.col("promotion_code").alias("ref_code"),
    F.col("start_date_parsed").alias("promo_start"),
    F.col("end_date_parsed").alias("promo_end")
)

orders = (
    orders.join(
        ref_promos,
        orders.promotion_code == F.col("ref_code"),
        "left"
    )
    .withColumn(
        "promotion_issue",
        F.when(missing("promotion_code"), "")
        .when(F.col("ref_code").isNull(), "PROMO_INCONNUE")
        .when(
            ~F.col("order_date_parsed").between(
                F.col("promo_start"), F.col("promo_end")
            ),
            "PROMO_HORS_VALIDITE"
        )
        .otherwise("")
    )
)

orders_silver, orders_quarantine, orders_audit = save_quality(
    "orders", orders, "order_id", read_bronze("orders").columns
)


orders: 44 = 40 Silver + 3 rejets + 1 copies


In [42]:
#order_items
items = read_bronze("order_items")

items = (
    items.withColumn(
        "quantity_parsed",
        F.expr("try_cast(quantity as int)")
    )
    .withColumn(
        "unit_price_parsed",
        F.expr("try_cast(unit_price as decimal(12,2))")
    )
    .join(
        orders_silver.select("order_id")
        .withColumn("order_found", F.lit(True)),
        on="order_id",
        how="left"
    )
    .join(
        products_silver.select("product_id")
        .withColumn("product_found", F.lit(True)),
        on="product_id",
        how="left"
    )
)

items = add_reasons(items, [
    ("ID_MANQUANT", missing("order_item_id")),
    ("COMMANDE_INCONNUE",
     missing("order_id") | F.col("order_found").isNull()),
    ("PRODUIT_INCONNU",
     missing("product_id") | F.col("product_found").isNull()),
    ("QUANTITE_INVALIDE",
     F.col("quantity_parsed").isNull() |
     (F.col("quantity_parsed") <= 0)),
    ("PRIX_INVALIDE",
     F.col("unit_price_parsed").isNull() |
     (F.col("unit_price_parsed") < 0))
])

items_silver, items_quarantine, items_audit = save_quality(
    "order_items", items, "order_item_id",
    read_bronze("order_items").columns
)

order_items: 86 = 80 Silver + 5 rejets + 1 copies
